# CCTV Safety — Stage 1 Kaggle continuation (YOLOv8n)

Historical notebook for the earlier 47-epoch YOLOv8n checkpoint attempt. A later Kaggle pilot reached 81/100 epochs; see `docs/project_status.md` and do not confuse the two artifacts. This notebook does **not** start training by default. The unified dataset remains provisional; review `data/processed/stage1_colab_bundle/validation_report.md` before authorizing any new run. Exactly six Stage 1 classes are used: `person`, `helmet`, `vest`, `fall`, `fire`, `smoke`; Fight is excluded.

**Do not use Run All.** Run the preparation/preflight cells first. The training guard remains `False` until the owner explicitly approves a new continuation.

## Kaggle inputs needed

Create **Private Kaggle Dataset(s)** and attach them with **Add Input**:

- `stage1_unified_colab.zip` from the local Stage 1 bundle (about 3.14 GiB)
- `yolov8n-20260927T090240Z-1-001.zip` from Downloads (contains `last.pt`, `best.pt`, `args.yaml`, and `results.csv`)

They can be separate private datasets. Kaggle may unpack uploaded ZIPs automatically, so the setup cells accept either the original ZIP or expanded files. For later sessions, also attach this notebook's saved output (Add Input → Notebook Output Files). The notebook selects the newest checkpoint by recorded epoch. Keep saved outputs in `/kaggle/working/stage1_runs`; extracted data is staged in `/kaggle/temp` so it is not duplicated into every saved output.

In [ ]:
from pathlib import Path
import csv, json, shutil, zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
TEMP_ROOT = Path('/kaggle/temp')
DATA_ZIP_NAME = 'stage1_unified_colab.zip'
CHECKPOINT_ZIP_NAME = 'yolov8n-20260927T090240Z-1-001.zip'
DATA_ROOT = TEMP_ROOT / 'stage1_unified'
DATA_YAML = DATA_ROOT / 'data_kaggle.yaml'
RUN_ROOT = WORK_ROOT / 'stage1_runs' / 'stage1_first_training'
MODEL_RUN_DIR = RUN_ROOT / 'yolov8n'
WEIGHTS_DIR = MODEL_RUN_DIR / 'weights'
LAST_PT = WEIGHTS_DIR / 'last.pt'
BEST_PT = WEIGHTS_DIR / 'best.pt'
DEVICE = '0,1'  # Kaggle T4 x2; change to '0' only if multi-GPU startup fails.
EXPECTED_NAMES = ['person', 'helmet', 'vest', 'fall', 'fire', 'smoke']

DATA_ARCHIVES = [p for p in INPUT_ROOT.rglob(DATA_ZIP_NAME) if p.is_file()]
DATA_MANIFESTS = [p for p in INPUT_ROOT.rglob('dataset_manifest.json') if p.is_file()]
CHECKPOINT_ARCHIVES = [p for p in INPUT_ROOT.rglob(CHECKPOINT_ZIP_NAME) if p.is_file()]
if len(DATA_ARCHIVES) > 1 or len(DATA_MANIFESTS) > 1 or len(CHECKPOINT_ARCHIVES) > 1:
    raise RuntimeError('Found duplicate Stage 1 inputs. Attach only one copy of each source.')
if not DATA_ARCHIVES and not DATA_MANIFESTS:
    raise FileNotFoundError('Attach the private Stage 1 dataset; expected its ZIP or extracted dataset_manifest.json.')
if not CHECKPOINT_ARCHIVES and not any(p.name == 'last.pt' for p in INPUT_ROOT.rglob('last.pt')):
    raise FileNotFoundError('Attach the private checkpoint dataset; expected its ZIP or expanded yolov8n/weights/last.pt.')
DATA_ARCHIVE = DATA_ARCHIVES[0] if DATA_ARCHIVES else None
DATA_SOURCE_ROOT = DATA_MANIFESTS[0].parent if DATA_MANIFESTS else None
CHECKPOINT_ARCHIVE = CHECKPOINT_ARCHIVES[0] if CHECKPOINT_ARCHIVES else None
print('Dataset source:', DATA_ARCHIVE or DATA_SOURCE_ROOT)
print('Checkpoint source:', CHECKPOINT_ARCHIVE or 'expanded checkpoint files under /kaggle/input')
print('Run outputs:', RUN_ROOT)

In [ ]:
# Safe extraction: prevent ZIP path traversal and verify archive CRCs while reading.
def safe_extract(zip_path, destination):
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            relative = Path(info.filename.replace('\\', '/'))
            target = (destination / relative).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Unsafe ZIP member path: {info.filename}')
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)

required = [DATA_ROOT / 'dataset_manifest.json', DATA_ROOT / 'data.yaml'] + [DATA_ROOT / 'metadata' / f'{s}.csv' for s in ('train', 'val', 'test')]
if not all(p.is_file() for p in required):
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    if DATA_ARCHIVE is not None:
        safe_extract(DATA_ARCHIVE, DATA_ROOT)
    else:
        shutil.copytree(DATA_SOURCE_ROOT, DATA_ROOT, dirs_exist_ok=True)
assert all(p.is_file() for p in required), 'Dataset extraction/copy is incomplete.'
manifest = json.loads((DATA_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
assert manifest['class_names'] == EXPECTED_NAMES, manifest.get('class_names')
assert int(manifest['total_images']) == 23169, manifest.get('total_images')
print('Dataset staged and manifest checks passed:', manifest['total_images'], 'images /', manifest['total_instances'], 'boxes')
print('Provisional:', manifest.get('provisional'), '| validation_only:', manifest.get('validation_only'), '| manifest training_approved:', manifest.get('training_approved'))

# Resolve absolute Kaggle paths in a fresh dataset YAML.
import yaml
data_config = {'path': str(DATA_ROOT.resolve()), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 6, 'names': EXPECTED_NAMES}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding='utf-8')
print('Kaggle data YAML:', DATA_YAML.read_text(encoding='utf-8'))

In [ ]:
# Full split, label, metadata, class, and leakage preflight.
splits = ('train', 'val', 'test')
groups, image_counts = {}, {}
class_counts = {name: 0 for name in EXPECTED_NAMES}
boundary_warnings = 0
valid_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in splits:
    images = [p for p in (DATA_ROOT / 'images' / split).glob('*') if p.is_file() and p.suffix.lower() in valid_suffixes]
    labels_dir = DATA_ROOT / 'labels' / split
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels_dir.glob('*.txt')}
    assert image_stems == label_stems, f'{split} image/label mismatch: missing={len(image_stems-label_stems)}, orphan={len(label_stems-image_stems)}'
    image_counts[split] = len(images)
    with (DATA_ROOT / 'metadata' / f'{split}.csv').open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == len(images), f'{split} metadata count mismatch'
    assert {r['image'] for r in rows} == {p.name for p in images}, f'{split} metadata filename mismatch'
    assert all(r['group_id'].strip() for r in rows), f'{split} has empty group_id'
    groups[split] = {r['group_id'] for r in rows}
    for image in images:
        for line_no, line in enumerate((labels_dir / f'{image.stem}.txt').read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            assert len(parts) == 5, f'{image.name}:{line_no}: malformed YOLO row'
            cid = int(parts[0])
            assert 0 <= cid < 6, f'{image.name}:{line_no}: invalid class {cid}'
            class_counts[EXPECTED_NAMES[cid]] += 1
            x, y, w, h = map(float, parts[1:])
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, f'{image.name}:{line_no}: invalid normalized box'
            boundary_warnings += int(x-w/2 < 0 or y-h/2 < 0 or x+w/2 > 1 or y+h/2 > 1)
for a, b in [('train','val'), ('train','test'), ('val','test')]:
    assert not (groups[a] & groups[b]), f'Group leakage between {a} and {b}'
assert image_counts == manifest['images_by_split'], (image_counts, manifest['images_by_split'])
assert class_counts == manifest['instances'], (class_counts, manifest['instances'])
print('DATA PREFLIGHT PASS')
print('Images by split:', image_counts)
print('Class instances:', class_counts)
print('Known boundary warnings:', boundary_warnings)

In [ ]:
# Stage the initial archive if Kaggle did not unpack it; also accept expanded files under /kaggle/input.
INITIAL_CHECKPOINT_ROOT = TEMP_ROOT / 'initial_colab_checkpoint'
if INITIAL_CHECKPOINT_ROOT.exists():
    shutil.rmtree(INITIAL_CHECKPOINT_ROOT)
if CHECKPOINT_ARCHIVE is not None:
    safe_extract(CHECKPOINT_ARCHIVE, INITIAL_CHECKPOINT_ROOT)

def recorded_epoch(results_csv):
    if not results_csv.is_file():
        return -1
    with results_csv.open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    return int(float(rows[-1]['epoch'])) if rows else -1

candidates = []
# Colab checkpoint, whether Kaggle exposed the ZIP or unpacked its yolov8n folder.
search_roots = [INITIAL_CHECKPOINT_ROOT] if CHECKPOINT_ARCHIVE is not None else []
search_roots.append(INPUT_ROOT)
for root in search_roots:
    for p in root.rglob('last.pt'):
        if p.parent.name == 'weights' and p.parent.parent.name == 'yolov8n':
            priority = 0 if root == INITIAL_CHECKPOINT_ROOT else 1
            candidates.append((recorded_epoch(p.parent.parent / 'results.csv'), priority, p))
# A checkpoint already present in this live session takes precedence if it is at least as recent.
if LAST_PT.is_file():
    candidates.append((recorded_epoch(MODEL_RUN_DIR / 'results.csv'), 2, LAST_PT))
assert candidates, 'No YOLOv8n last.pt found in the attached private inputs or prior notebook output.'
source_epoch, _, source_last = max(candidates, key=lambda item: (item[0], item[1]))
source_model_dir = source_last.parent.parent
assert source_epoch >= 0, f'No results.csv alongside checkpoint: {source_last}'
assert (source_model_dir / 'args.yaml').is_file(), f'Missing args.yaml beside {source_last}'
assert (source_model_dir / 'weights' / 'best.pt').is_file(), 'Missing best.pt beside source checkpoint'
if source_model_dir.resolve() != MODEL_RUN_DIR.resolve():
    shutil.copytree(source_model_dir, MODEL_RUN_DIR, dirs_exist_ok=True)
assert LAST_PT.is_file() and BEST_PT.is_file()
print('Resume source:', source_last)
print('Latest recorded epoch:', source_epoch, '(verify checkpoint epoch in the next cell before enabling the guard)')
print('Checkpoint size MiB:', round(LAST_PT.stat().st_size / 1024**2, 2))
print('Run args:')
print((MODEL_RUN_DIR / 'args.yaml').read_text(encoding='utf-8'))
print('Output checkpoint:', LAST_PT)

In [ ]:
# GPU preflight only. Choose GPU T4 x2 in Kaggle Settings before executing this cell.
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Stop: enable the Kaggle GPU accelerator first.'
assert torch.cuda.device_count() >= 2, f'DEVICE={DEVICE} requests two GPUs, but only {torch.cuda.device_count()} are visible.'
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

## Verify the pinned training library

The Colab checkpoint was made with Ultralytics 8.4.163. Check the Kaggle image first. If the version differs, use the optional install cell below with Internet enabled, then restart the session and rerun preparation/preflight. Do not upgrade PyTorch/CUDA packages.

In [ ]:
import importlib.metadata
try:
    ultralytics_version = importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    ultralytics_version = 'not installed'
print('Installed Ultralytics:', ultralytics_version)
print('Required for exact checkpoint continuity: 8.4.163')

In [ ]:
%pip install -q --no-deps ultralytics==8.4.163

## Training gate — leave disabled during preparation

The checkpoint shows YOLOv8n training through epoch 47 of 100 with `batch=16`, `imgsz=640`, and `workers=2`. The code below resumes that run, retaining optimizer/scheduler state, and redirects output to Kaggle's persistent notebook output directory. It requests both T4s while keeping total batch 16. Kaggle's T4 x2 does not guarantee 2× speed; compare actual epoch time.

Before resuming, confirm the preflight passed, the newest checkpoint is attached, the library version is exactly 8.4.163, and you have reviewed the provisional dataset limitations.

In [ ]:
TRAINING_APPROVED = False  # Change to True only when you choose to resume training on Kaggle.
if not TRAINING_APPROVED:
    print('Training remains disabled. Set TRAINING_APPROVED=True only when you explicitly decide to resume.')
elif source_epoch >= 100 or (MODEL_RUN_DIR / 'TRAINING_COMPLETE').is_file():
    print('YOLOv8n already has a completed-run marker or 100 recorded epochs; refusing to train it again.')
else:
    from ultralytics import YOLO
    import importlib.metadata
    current_ultralytics = importlib.metadata.version('ultralytics')
    assert current_ultralytics == '8.4.163', f'Expected Ultralytics 8.4.163, got {current_ultralytics}'
    assert LAST_PT.is_file() and BEST_PT.is_file()
    model = YOLO(str(LAST_PT))
    ckpt = getattr(model, 'ckpt', None) or {}
    completed_epochs = int(ckpt.get('epoch', -1)) + 1
    print(f'Checkpoint metadata: {completed_epochs} completed epochs; results.csv records epoch {source_epoch}.')
    model.train(
        resume=True, data=str(DATA_YAML), device=DEVICE, batch=16, imgsz=640, workers=2,
        save_dir=str(MODEL_RUN_DIR),
    )
    (MODEL_RUN_DIR / 'TRAINING_COMPLETE').write_text('YOLOv8n resume completed.', encoding='utf-8')
    best_model = YOLO(str(BEST_PT))
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', device='0')
    result = {
        'model': 'yolov8n', 'checkpoint_completed_epochs_before_resume': completed_epochs,
        'map50': float(test_metrics.box.map50), 'map50_95': float(test_metrics.box.map),
        'precision': float(test_metrics.box.mp), 'recall': float(test_metrics.box.mr),
        'class_names': list(best_model.names.values()), 'dataset_manifest': str(DATA_ROOT / 'dataset_manifest.json'),
        'provisional_dataset': True,
    }
    (RUN_ROOT / 'yolov8n_test_metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print('Completed and test metrics saved:', result)

## Preserve a checkpoint between Kaggle sessions

Ultralytics saves `last.pt` after each completed epoch under `/kaggle/working/stage1_runs/...`. Before ending a session, use **Save Version → Quick Save → Advanced Settings → save output files**. In the next session, attach that saved notebook output through **Add Input → Notebook Output Files**; the preflight cell will choose the candidate with the latest `results.csv` epoch. Do not use **Save & Run All** just to preserve checkpoints, because it reruns cells. Confirm the output version finished saving before closing the session. The dataset itself is extracted under `/kaggle/temp`, so it is not copied into saved outputs.